This notebook demonstrates how to use the BiomechZoo library to analyze movement data collected from IMU sensors.
The analysis includes data preprocessing, event detection, gait cycle partitioning, normalization, and visualization of the results.

Data source: The data used in this notebook is from the "Losing" dataset, which contains IMU data collected from participants performing
various walking tasks. The data were modified for the purposes of this tutorial.

The full dataset is available at: https://doi.org/10.6084/m9.figshare.c.5758997

The dataset used here is avaialbe in https://github.com/mcgillmotionlab/biomechzoo/tree/main/data/losing/raw

The dataset is described in the following paper: Losing, V., Hasenjäger, M. A Multi-Modal Gait Database of Natural Everyday-Walk in an Urban Environment. Sci Data 9, 473 (2022). https://doi.org/10.1038/s41597-022-01580-3

In [15]:
# import relevant biomech modules
%load_ext autoreload
%autoreload 2
import os
from biomechzoo.biomechzoo import BiomechZoo
from biomechzoo.utils.find_repo_root import find_repo_root

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
# Step 0: get raw data folder and initialize biomechzoo object
project_root = find_repo_root(os.getcwd())
fld_raw_data = os.path.join(project_root, 'data', 'losing', 'raw')
inplace = False  # tells biomechzoo to create new folders instead of writing to the original folder
verbose = 2 # minimal output to the user
name_excludes = ['section_1', 'section_3'] # these trials won't be processed
bmech = BiomechZoo(fld_raw_data, verbose=verbose, inplace=inplace)

BiomechZoo initialized
verbosity set to: 2
root processing folder set to: /Users/phild/Code/biomechzoo/data/losing/raw
Processing mode: backup (inplace=False)(each step will be applied to a new folder)


In [17]:
# step 1: convert to zoo
# frequency is 60 Hz as per the database documentation
bmech.table2zoo(extension='csv', freq=60, out_folder='1-csv2zoo')

converting .csv to zoo for /Users/phild/Code/biomechzoo/data/losing/raw/id14_courseB_50cycles.csv
converting .csv to zoo for /Users/phild/Code/biomechzoo/data/losing/raw/id16_courseA_50cycles.csv
converting .csv to zoo for /Users/phild/Code/biomechzoo/data/losing/raw/id19_courseC_50cycles.csv
table2zoo process complete for 3 file(s) in 0.74 secs
all files saved to: /Users/phild/Code/biomechzoo/data/losing/1-csv2zoo


In [18]:
# step 2 clean
ch_remove = ['walk_mode', 'walk_orientation',
             'time_orig']
bmech.removechannel(ch=ch_remove, out_folder='2-clean')

removing 3 channels for /Users/phild/Code/biomechzoo/data/losing/1-csv2zoo/id14_courseB_50cycles.zoo
removing 3 channels for /Users/phild/Code/biomechzoo/data/losing/1-csv2zoo/id16_courseA_50cycles.zoo
removing 3 channels for /Users/phild/Code/biomechzoo/data/losing/1-csv2zoo/id19_courseC_50cycles.zoo
removechannel process complete for 3 file(s) in 0.10 secs
all files saved to: /Users/phild/Code/biomechzoo/data/losing/2-clean


In [19]:
# inspect the file
bmech.file_info()

File: /Users/phild/Code/biomechzoo/data/losing/2-clean/id16_courseA_50cycles.zoo
Size: 1065456 bytes
Video frequency: 60 Hz
Analog frequency: {} Hz
Channels (40): time, acceleration_Pelvis_x, acceleration_Pelvis_y, acceleration_Pelvis_z, acceleration_RightLowerLeg_x, acceleration_RightLowerLeg_y, acceleration_RightLowerLeg_z, acceleration_RightFoot_x, acceleration_RightFoot_y, acceleration_RightFoot_z, acceleration_LeftLowerLeg_x, acceleration_LeftLowerLeg_y, acceleration_LeftLowerLeg_z, acceleration_LeftFoot_x, acceleration_LeftFoot_y, acceleration_LeftFoot_z, angularVelocity_Pelvis_x, angularVelocity_Pelvis_y, angularVelocity_Pelvis_z, angularVelocity_RightLowerLeg_x, angularVelocity_RightLowerLeg_y, angularVelocity_RightLowerLeg_z, angularVelocity_RightFoot_x, angularVelocity_RightFoot_y, angularVelocity_RightFoot_z, angularVelocity_LeftLowerLeg_x, angularVelocity_LeftLowerLeg_y, angularVelocity_LeftLowerLeg_z, angularVelocity_LeftFoot_x, angularVelocity_LeftFoot_y, angularVelocity_

In [20]:
# step 3: get foot strikes from Kielmat algorithm
bmech.compute_magnitude(ch_1='acceleration_Pelvis_x', ch_2='acceleration_Pelvis_y', ch_3='acceleration_Pelvis_z', ch_new='acceleration_Pelvis_magnitude', out_folder='3-processing', inplace=False)

bmech.addevent(event_type='kielmat_fs', event_name='foot_strike', ch='acceleration_Pelvis_magnitude', out_folder='3-processing', inplace=True)

compute magnitude from channels acceleration_Pelvis_x, acceleration_Pelvis_y, acceleration_Pelvis_z for /Users/phild/Code/biomechzoo/data/losing/2-clean/id14_courseB_50cycles.zoo
compute magnitude from channels acceleration_Pelvis_x, acceleration_Pelvis_y, acceleration_Pelvis_z for /Users/phild/Code/biomechzoo/data/losing/2-clean/id16_courseA_50cycles.zoo
compute magnitude from channels acceleration_Pelvis_x, acceleration_Pelvis_y, acceleration_Pelvis_z for /Users/phild/Code/biomechzoo/data/losing/2-clean/id19_courseC_50cycles.zoo
compute_magnitude process complete for 3 file(s) in 0.04 secs
all files saved to: /Users/phild/Code/biomechzoo/data/losing/3-processing
adding event kielmat_fs to channel acceleration_Pelvis_magnitude for /Users/phild/Code/biomechzoo/data/losing/3-processing/id14_courseB_50cycles.zoo
adding event kielmat_fs to channel acceleration_Pelvis_magnitude for /Users/phild/Code/biomechzoo/data/losing/3-processing/id16_courseA_50cycles.zoo
adding event kielmat_fs to ch

In [21]:
# step 4: partition by gait cycle and normalize
bmech.split_trial_by_gait_cycle(first_event_name='foot_strike_1', out_folder='4-split', inplace=False)
bmech.normalize()

splitting by gait cycle from foot_strike_1 to foot_strike_2 for /Users/phild/Code/biomechzoo/data/losing/3-processing/id14_courseB_50cycles.zoo
splitting by gait cycle from foot_strike_2 to foot_strike_3 for /Users/phild/Code/biomechzoo/data/losing/3-processing/id14_courseB_50cycles.zoo
splitting by gait cycle from foot_strike_3 to foot_strike_4 for /Users/phild/Code/biomechzoo/data/losing/3-processing/id14_courseB_50cycles.zoo
splitting by gait cycle from foot_strike_4 to foot_strike_5 for /Users/phild/Code/biomechzoo/data/losing/3-processing/id14_courseB_50cycles.zoo
splitting by gait cycle from foot_strike_5 to foot_strike_6 for /Users/phild/Code/biomechzoo/data/losing/3-processing/id14_courseB_50cycles.zoo
splitting by gait cycle from foot_strike_6 to foot_strike_7 for /Users/phild/Code/biomechzoo/data/losing/3-processing/id14_courseB_50cycles.zoo
splitting by gait cycle from foot_strike_7 to foot_strike_8 for /Users/phild/Code/biomechzoo/data/losing/3-processing/id14_courseB_50cyc

In [22]:
# step 7 plot all files
# import statements
from biomechzoo.ensembler.ensembler import Ensembler
from biomechzoo.ensembler.plot_spec import PlotSpec
from biomechzoo.ensembler.renderers import IndividualLinesRenderer, MeanSDRenderer, CompositeRenderer
from biomechzoo.ensembler.helpers import ConditionSpec, ConditionSource

spec = ConditionSpec(source = ConditionSource.BETWEEN,
                     conditions = ['running'])
channels = ['jointAngle_jRightKnee_z']
rows = 1
cols = 2

lines_and_events = CompositeRenderer(IndividualLinesRenderer())     # within stuff

fig = (
    Ensembler(in_folder=bmech.in_folder,  channels=channels, n_rows=rows,  n_cols=cols, condition_spec=spec)
    .add_subplot(PlotSpec(channel='acc_mag', condition = 'running', companions = [], row=1, col=1, renderer=lines_and_events))
    .add_subplot(PlotSpec(channel='acc_mag', condition = 'running', companions = [], row=1, col=2, renderer=MeanSDRenderer()))
    .build(title=' running')
)
fig.show()


